# Notebook 7 — Next-month pick count per SKU

**Church & Dwight | York Distribution Center**

**What it forecasts.** The number of times each SKU will be picked over the next month. One row in the pick files is one pick, so the target is a count of rows: `n_lines` in your `sku_day_panel.csv` (every customer-order pick row, all zones and units of measure). The panel is built from both raw files (current + archive), so it already contains all the picks.

**How it works, in one paragraph.** For every SKU the notebook trains LightGBM to predict next month's pick count *as a multiplier on last month's pick count* (so SKUs with steady demand stay near last month's level and the model only learns what changes). It blends that with plain "last 28 days", and **chooses the blend separately for each consistency group** (Steady / Variable / Erratic / Rare), because those groups behave very differently. Everything is tested on 12 rolling origins; the latest 28 days are locked.

| Decision | What the notebook does | Why |
|---|---|---|
| **Target** | all picks per SKU (`n_lines`); the split into case picks / pallet pulls / other is used as *context* | This is the count you asked for. About 38% of picks are pick-face case picks, 58% reserve pallet pulls. The output also gives the expected case picks, which is the part the slotting algorithm counts. |
| **Horizon** | 28 days, one total per SKU (`HORIZON_DAYS`, change to 30 or 31 if you like) | Matches the monthly re-slotting. Four whole weeks keeps the weekday mix equal. |
| **Groups** | by **consistency**: the variation of a SKU's weekly pick totals over the last 13 weeks, plus a Rare group (sells on under 5% of days) | In a test, the error ranges from about 20% (Steady) to 70% (Erratic) to over 100% (Rare). One setting for all of them is a poor fit. |
| **Models** | one global LightGBM ratio model + simple baselines, **blend weight picked per group** with leave-one-origin-out so the reported score is not flattered | Separate models per group were not tried here: with about a year of data each group would have too few rows. Groups differ in *how much to trust the model*, which the blend weight captures. |
| **Seasonality** | features for "what happened in this window a year ago" are built in and switch on automatically once the data covers more than a year | You do not need to change any code when the second year arrives; just re-run the preprocessing and this notebook. |
| **Uncertainty** | an 80% range per SKU, from how wrong the forecast was in the development origins for SKUs of that group and size | A single number hides that Erratic SKUs are much less predictable than Steady ones. |

**Protocol.** Development origins: every 14 days for about 6 months. Locked test: the final 28 days, scored only if you set `RUN_LOCKED_TEST = True`. These are forecasts of **picks, not customer demand** (the order-line extract has not arrived).

> **How to run:** edit the paths in the next cell, then Kernel → Restart & Run All. If LightGBM is missing: `%pip install lightgbm` and restart.

In [ ]:
%pip install lightgbm

In [ ]:
import sys
print(sys.executable)

In [ ]:

from pathlib import Path
import time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- paths (edit these)
DATA_DIR = Path("/Users/ameli/projects/reslot-forecast/data/processed")  # your data folder
PANEL_FILE = DATA_DIR / "sku_day_panel.csv"          # your preprocessing output (.csv.gz also works)
OUTPUT = DATA_DIR / "forecast output" / "pick_count_forecast_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- what we forecast
TARGET_COL = "n_lines"                       # picks per SKU per day: every pick row (one row = one pick)
CONTEXT_COLS = {"CASE": "case_pick_lines",   # pick-face case picks     (used as predictors and for the case-pick output)
                "PULL": "pallet_pull_lines", # reserve pallet pulls
                "OTH":  "other_lines"}       # everything else
HORIZON_DAYS = 28                            # one re-slotting period (use 30 or 31 for a calendar month)

# ---------------------------------------------------------------- grouping by consistency
RARE_MAX_ACTIVE_RATE = 0.05    # picked on under 5% of its days so far  -> Rare
CV_STEADY   = 0.40             # weekly-total variation below this        -> Steady
CV_VARIABLE = 0.80             # below this                               -> Variable, otherwise Erratic
CV_WEEKS = 13                  # weeks of history used for the variation measure

# ---------------------------------------------------------------- evaluation settings
MIN_HISTORY = 28               # days of history before a SKU is forecast
TRAIN_ORIGIN_STEP = 7          # training snapshots every 7 days
EVAL_STEP = 14                 # days between development origins
N_EVAL_ORIGINS = 12            # about 6 months of development origins
MOVER_TOP_FRAC = 0.10          # "movers" = the 10% of SKUs whose pick count changed most
RANDOM_SEED = 42

# ---------------------------------------------------------------- model / blend settings
BASE_FLOOR = 0.25              # picks per period used as the starting level for SKUs with no recent picks
W_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)   # blend weights on the LightGBM ratio model; 0 = last 28 days only
EXTRA_METHODS = ("Last 90 days", "All-history average")   # other simple options each group may choose instead of a blend
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                  colsample_bytree=0.8, subsample=0.8, subsample_freq=1, reg_lambda=5.0,
                  random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1)
DROP_FEATURES = []             # e.g. ["PULL28", "PULL90"] to test without the context features
SEASONAL_LAG_DAYS = 364        # last-year features switch on automatically when the history is long enough
CALENDAR_FEATURES = False      # month / holiday-season of the forecast window; turn on once you have 2+ years

# ---------------------------------------------------------------- output flags
INTERVAL = (0.10, 0.90)        # the range reported per SKU
MOVER_THRESHOLD = 0.25         # flag "up"/"down" if forecast differs from last 28 days by >= 25% ...
MOVER_MIN_PICKS = 20           # ... and the SKU had at least this many picks in the last 28 days
RUN_LOCKED_TEST = False        # set True ONCE, after the method is settled, to score the locked last period

H = int(HORIZON_DAYS)
GROUPS = ["Steady", "Variable", "Erratic", "Rare"]
print(f"Target {TARGET_COL} (all picks) | horizon {H} days | {N_EVAL_ORIGINS} development origins every {EVAL_STEP} days")

## 1. Load the daily SKU panel
One row per SKU per full day. `n_lines` is the pick count. It equals `case_pick_lines + pallet_pull_lines + other_lines` on every row (checked below). Days are zero-filled after a SKU's first pick.

In [ ]:
def find_panel():
    for p in (PANEL_FILE, PANEL_FILE.with_suffix(PANEL_FILE.suffix + ".gz"), PANEL_FILE.with_suffix(".csv.gz")):
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find the panel at {PANEL_FILE} (or .gz). Edit PANEL_FILE in the settings cell.")

panel_path = find_panel()
print("Reading", panel_path)
use = ["item_id", "date", "source", TARGET_COL] + [c for c in CONTEXT_COLS.values() if c != TARGET_COL]
panel = pd.read_csv(panel_path, usecols=use, dtype={"item_id": str}, parse_dates=["date"])
SOURCE_TAG = str(panel["source"].iloc[0])
assert (panel[[TARGET_COL] + list(CONTEXT_COLS.values())] >= 0).all().all(), "Negative counts?"
assert not panel.duplicated(["item_id", "date"]).any(), "Duplicate SKU-days?"
if TARGET_COL == "n_lines":
    gap = (panel[list(CONTEXT_COLS.values())].sum(axis=1) - panel[TARGET_COL]).abs().max()
    print("max |picks - (case + pallet + other)| on any row:", gap)
tot = panel[[TARGET_COL] + list(CONTEXT_COLS.values())].sum()
print("SKUs:", panel.item_id.nunique(), "| days:", panel.date.nunique(), "|", panel.date.min().date(), "to", panel.date.max().date(), "| source tag:", SOURCE_TAG)
print((tot / tot[TARGET_COL]).rename("share of all picks").round(3).to_frame().T.to_string())

## 2. SKU × day matrices, running sums, and the split

Everything the model sees at an origin day `o` uses days up to and including `o` only. Running sums make every rolling window a subtraction.

**The split.** Locked test = the final 28 days. Development origins = every 14 days going back from just before it, so every development outcome ends before the locked window begins. A model evaluated at origin `e` is trained only on origins whose 28-day outcome was already known at `e`.

In [ ]:
skus = pd.Index(sorted(panel.item_id.unique()), name="SKU")
dates = pd.date_range(panel.date.min(), panel.date.max(), freq="D", name="DATE")
si = pd.Series(np.arange(len(skus)), index=skus)
di = pd.Series(np.arange(len(dates)), index=dates)
rows = si.loc[panel.item_id].to_numpy(); cols = di.loc[panel.date].to_numpy()
n_sku, n_days = len(skus), len(dates)

def to_matrix(col):
    a = np.zeros((n_sku, n_days), dtype=np.float64)
    np.add.at(a, (rows, cols), panel[col].to_numpy(np.float64))
    return a
tgt = to_matrix(TARGET_COL)
ctx = {k: to_matrix(c) for k, c in CONTEXT_COLS.items()}

first_seen = np.full(n_sku, n_days, dtype=np.int32)          # first day with any pick
r_, c_ = np.nonzero(tgt > 0); np.minimum.at(first_seen, r_, c_)

def cum(a): return np.concatenate([np.zeros((n_sku, 1)), np.cumsum(a, axis=1)], axis=1)
CS_T, CS_T2, CS_A = cum(tgt), cum(tgt ** 2), cum((tgt > 0).astype(float))
CS_X = {k: cum(v) for k, v in ctx.items()}
last_active = np.maximum.accumulate(np.where(tgt > 0, np.arange(n_days)[None, :], -1), axis=1)

def wsum(CS, o, w):                       # sum over days o-w+1 .. o
    return CS[:, o + 1] - CS[:, max(o + 1 - w, 0)]

test_origin = n_days - 1 - H
eval_origins = [test_origin - H - k * EVAL_STEP for k in range(N_EVAL_ORIGINS)][::-1]
final_origin = n_days - 1
train_grid = [o for o in range(MIN_HISTORY - 1, n_days - H, TRAIN_ORIGIN_STEP)]
assert eval_origins[0] - H >= MIN_HISTORY + 2 * TRAIN_ORIGIN_STEP, "Not enough history: lower N_EVAL_ORIGINS or EVAL_STEP."
print("Calendar:", dates[0].date(), "to", dates[-1].date())
print("Development origins:", dates[eval_origins[0]].date(), "...", dates[eval_origins[-1]].date(), f"({len(eval_origins)})")
print("LOCKED test: origin", dates[test_origin].date(), "-> outcome", dates[test_origin + 1].date(), "to", dates[-1].date(), "(not used to choose anything)")
print("Real forecast origin:", dates[final_origin].date(), "-> next", H, "days")

## 3. Groups, features and baselines

**Groups (by consistency).** At each origin, using only the past:

* **Rare:** picked on under 5% of its days so far.
* Otherwise the **coefficient of variation of the weekly pick totals** over the last 13 weeks (standard deviation ÷ mean; only complete weeks after the SKU's first pick). **Steady** < 0.4, **Variable** < 0.8, **Erratic** otherwise.

A SKU can move between groups from one month to the next; the group is recomputed at every origin.

**Features** describe each SKU's recent level (7 to 90 days), trend, variability, how often it sells, the mix of case picks vs pallet pulls, and its weekly variation. There is no SKU identifier and no month feature by default. **Last-year features** (what this SKU's next-28-day count was relative to its prior 28 days one year ago) are created when the history is long enough and ignored otherwise.

**Baselines:** last 28 days (average × horizon), last 90 days, their average, and the SKU's whole-history average.

In [ ]:
def business_days(o):
    d = pd.date_range(dates[o] + pd.Timedelta(days=1), periods=H)
    return int((d.dayofweek < 5).sum())

def eligible(o):
    return np.flatnonzero((first_seen <= o) & ((o - first_seen + 1) >= MIN_HISTORY))

def weekly_cv(o, el):
    """CV of weekly pick totals over the last CV_WEEKS weeks; only complete weeks after the SKU's first pick."""
    W, V = [], []
    for k in range(CV_WEEKS):
        end = o - 7 * (CV_WEEKS - 1 - k); start = end - 6
        if end < 0:
            W.append(np.zeros(len(el))); V.append(np.zeros(len(el), bool)); continue
        W.append(CS_T[el, end + 1] - CS_T[el, max(start, 0)])
        V.append((start >= 0) & (first_seen[el] <= start))
    W, V = np.stack(W, 1), np.stack(V, 1)
    Wm = np.where(V, W, np.nan)
    with np.errstate(all="ignore"):
        mu, sd = np.nanmean(Wm, axis=1), np.nanstd(Wm, axis=1)
    return np.where(V.sum(1) >= 3, sd / np.maximum(mu, 1e-9), np.nan), V.sum(1)

def assign_group(rate, cv):
    g = np.where(cv < CV_STEADY, "Steady", np.where(cv < CV_VARIABLE, "Variable", "Erratic"))
    g = np.where(np.isnan(cv), "Erratic", g)            # too little history to judge: treat with caution
    return np.where(rate < RARE_MAX_ACTIVE_RATE, "Rare", g)

def make_features(o):
    el = eligible(o); f = {}
    for w in (7, 14, 28, 56, 90):
        f[f"T{w}"] = wsum(CS_T, o, w)[el]
    for k, CS in CS_X.items():
        f[f"{k}28"], f[f"{k}90"] = wsum(CS, o, 28)[el], wsum(CS, o, 90)[el]
    f["CASE_SHARE"] = f["CASE28"] / (f["T28"] + 1); f["PULL_SHARE"] = f["PULL28"] / (f["T28"] + 1)
    f["ACTIVE28"], f["ACTIVE90"] = wsum(CS_A, o, 28)[el] / 28, wsum(CS_A, o, 90)[el] / 90
    m28 = f["T28"] / 28
    f["STD28"] = np.sqrt(np.maximum(wsum(CS_T2, o, 28)[el] / 28 - m28 ** 2, 0))
    f["R7_28"] = (f["T7"] / 7) / (m28 + 0.05); f["R28_90"] = m28 / (f["T90"] / 90 + 0.05)
    la = last_active[el, o]
    f["DAYS_SINCE"] = np.where(la >= 0, o - la, o - first_seen[el] + 1).astype(float)
    f["AGE"] = (o - first_seen[el] + 1).astype(float)
    f["BUS_DAYS"] = np.full(len(el), business_days(o), dtype=float)
    cv, nwk = weekly_cv(o, el); f["WEEKLY_CV"], f["WEEKS_USED"] = cv, nwk.astype(float)
    # last year's change for this window: (next-H picks a year ago) / (prior-28-day rate a year ago x H)
    ly = o - SEASONAL_LAG_DAYS
    if ly - 28 >= 0:
        then_base = (CS_T[el, ly + 1] - CS_T[el, ly - 27]) / 28 * H
        then_next = CS_T[el, ly + H + 1] - CS_T[el, ly + 1]
        f["LY_CHANGE"] = np.where(then_base > 0, then_next / np.maximum(then_base, 1e-9), np.nan)
    else:
        f["LY_CHANGE"] = np.full(len(el), np.nan)
    if CALENDAR_FEATURES:
        mid = dates[o] + pd.Timedelta(days=H // 2)
        f["WINDOW_MONTH"] = np.full(len(el), mid.month, dtype=float)
        f["HOLIDAY_SEASON"] = np.full(len(el), float(mid.month in (11, 12)))
    X = pd.DataFrame(f).drop(columns=DROP_FEATURES, errors="ignore")
    base = f["T28"] / 28 * H
    rate = wsum(CS_A, o, 10 ** 6)[el] / (o - first_seen[el] + 1)
    return dict(el=el, X=X, base=base, rate=rate, group=assign_group(rate, cv),
                t90=f["T90"] / 90 * H, hist_avg=wsum(CS_T, o, 10 ** 6)[el] / (o - first_seen[el] + 1) * H)

def outcome(o):
    return tgt[:, o + 1:o + 1 + H].sum(axis=1)

t0 = time.time()
need = sorted(set(train_grid) | set(eval_origins) | {test_origin, final_origin})
FE = {o: make_features(o) for o in need}
dead = [c for c in FE[need[0]]["X"].columns if all(FE[o]["X"][c].isna().all() for o in need)]
if dead:
    print("Switched off (no data yet):", dead, "(the last-year feature turns on once the history is longer than", SEASONAL_LAG_DAYS + 28 + H, "days)")
    for o in need: FE[o]["X"] = FE[o]["X"].drop(columns=dead)
print(f"Features built for {len(need)} origins in {time.time() - t0:.1f}s | {FE[need[-1]]['X'].shape[1]} features")
g = pd.Series(FE[eval_origins[-1]]["group"]).value_counts().reindex(GROUPS).to_frame("SKUs")
a_ = outcome(eval_origins[-1])[FE[eval_origins[-1]]["el"]]
g["share of next-28-day picks"] = pd.Series(a_).groupby(FE[eval_origins[-1]]["group"]).sum().reindex(GROUPS) / a_.sum()
print(f"\nGroups at origin {dates[eval_origins[-1]].date()} (the latest development origin):"); display(g.round(3))

## 4. The LightGBM models

* **Ratio model (the main one):** Poisson loss on the 28-day pick total, starting from last month's level (an offset), so it learns a *multiplier*. A steady SKU's multiplier stays near 1.
* **Level model:** the same without the starting level, kept as a reference.

For an evaluation origin `e`, models are trained only on training origins `o` with `o + H <= e`.

In [ ]:
def fit_models(e):
    ors = [o for o in train_grid if o + H <= e]
    X = pd.concat([FE[o]["X"] for o in ors], ignore_index=True)
    y = np.concatenate([outcome(o)[FE[o]["el"]] for o in ors])
    b = np.concatenate([FE[o]["base"] for o in ors])
    lvl = LGBMRegressor(objective="poisson", **LGB_PARAMS).fit(X, y)
    rat = LGBMRegressor(objective="poisson", **LGB_PARAMS).fit(X, y, init_score=np.log(np.maximum(b, BASE_FLOOR)))
    return dict(level=lvl, ratio=rat, n_rows=len(X), n_origins=len(ors))

def predict_models(models, fe):
    lvl = np.maximum(models["level"].predict(fe["X"]), 0)
    rat = np.exp(models["ratio"].predict(fe["X"], raw_score=True) + np.log(np.maximum(fe["base"], BASE_FLOOR)))
    return lvl, rat

def all_predictions(e, models):
    fe = FE[e]; lvl, rat = predict_models(models, fe)
    out = {"Last 28 days": fe["base"], "Last 90 days": fe["t90"], "Avg of last 28 & 90": (fe["base"] + fe["t90"]) / 2,
           "All-history average": fe["hist_avg"], "LightGBM level": lvl, "LightGBM ratio": rat}
    for w in W_GRID:
        out[f"Blend {w:.2f}"] = w * rat + (1 - w) * fe["base"]
    return out

t0 = time.time()
m_test = fit_models(eval_origins[-1])
print(f"One fit: {m_test['n_rows']:,} training rows from {m_test['n_origins']} origins ({time.time() - t0:.1f}s)")
imp = pd.Series(m_test["ratio"].booster_.feature_importance("gain"), index=FE[eval_origins[-1]]["X"].columns).sort_values(ascending=False)
print("Most used features in the ratio model (latest development fit):")
display((imp / imp.sum()).round(3).head(8).to_frame("share of gain"))

## 5. Rolling-origin evaluation (development origins only)

At each development origin: train on what was known then, forecast the next 28 days for every eligible SKU, compare with what happened. Reported per origin and then averaged:

* **WAPE:** total absolute error ÷ total actual picks. **Bias:** total forecast vs actual.
* **Top-5% / top-20% hit rate and top-100 overlap:** of the SKUs that were really the busiest, how many did the forecast put in its own top group?
* **Movers:** the 10% of SKUs whose pick count changed most vs last 28 days: error on just them, and how often the forecast moved the same direction (the last-28-days baseline never moves, so it scores 0 by construction).

In [ ]:
def score_one(p, a, base):
    n = len(a); tot = max(a.sum(), 1e-9)
    def hit(k):
        k = max(int(k), 1)
        return len(set(np.argsort(-a, kind="stable")[:k]) & set(np.argsort(-p, kind="stable")[:k])) / k
    pos = a > 0
    rho = spearmanr(p[pos], a[pos])[0] if pos.sum() > 5 else np.nan
    mv = np.argsort(-np.abs(a - base), kind="stable")[:max(int(n * MOVER_TOP_FRAC), 1)]
    return dict(WAPE=np.abs(p - a).sum() / tot, bias=(p.sum() - a.sum()) / tot,
                top5=hit(round(n * 0.05)), top20=hit(round(n * 0.20)), top100=hit(100), spearman=rho,
                mover_WAPE=np.abs(p - a)[mv].sum() / max(a[mv].sum(), 1e-9),
                mover_dir=float(np.mean(np.sign(np.round(p - base, 6))[mv] == np.sign(a - base)[mv])))

t0 = time.time()
STORE = {}                                  # origin -> dict(a, base, group, preds)
for e in eval_origins:
    models = fit_models(e); fe = FE[e]
    STORE[e] = dict(a=outcome(e)[fe["el"]], base=fe["base"], group=fe["group"], preds=all_predictions(e, models))
print(f"{len(eval_origins)} origins fitted and forecast in {time.time() - t0:.0f}s")

recs = []
for e, s in STORE.items():
    for name, p in s["preds"].items():
        r = score_one(p, s["a"], s["base"]); r.update(origin=dates[e], method=name); recs.append(r)
dev = pd.DataFrame(recs)
cols_show = ["WAPE", "bias", "top5", "top20", "top100", "spearman", "mover_WAPE", "mover_dir"]
summary = dev.groupby("method")[cols_show].mean()
l28 = dev[dev.method == "Last 28 days"].set_index("origin").WAPE
summary["beats Last 28 (share of origins)"] = [(dev[dev.method == m].set_index("origin").WAPE < l28).mean() for m in summary.index]
order = ["Last 28 days", "Last 90 days", "Avg of last 28 & 90", "All-history average", "LightGBM level", "LightGBM ratio"] + [f"Blend {w:.2f}" for w in W_GRID]
summary = summary.loc[order]
pd.options.display.float_format = "{:.3f}".format
display(summary)

**Reading the table.** Compare everything with *Last 28 days*, which is what is effectively used today. `Blend 0.00` equals Last 28 days and `Blend 1.00` equals the ratio model, so the Blend rows show the trade-off between them. A difference under about one point of WAPE is within the origin-to-origin spread, so also look at the last column.

## 6. Which method for which group?

Error is computed **per group**, summed over the development origins. Each group then picks its own method from the menu (a blend at each weight, plus the extra simple methods). The choice is scored **leave-one-origin-out**: for each origin, the choice is made using the *other* origins only, so the reported "Per-group choice" score is not flattered by choosing and scoring on the same data.

In [ ]:
MENU = [f"Blend {w:.2f}" for w in W_GRID] + list(EXTRA_METHODS)
grp = []
for e, s in STORE.items():
    for g_ in GROUPS:
        m = s["group"] == g_
        row = dict(origin=e, group=g_, n=int(m.sum()), actual=s["a"][m].sum())
        for meth in MENU + ["Last 28 days", "LightGBM ratio"]:
            row[meth] = np.abs(s["preds"][meth] - s["a"])[m].sum()
        grp.append(row)
grp = pd.DataFrame(grp)

def choose(df):
    """per group: the menu method with the smallest total absolute error"""
    tot = df.groupby("group")[MENU].sum()
    return tot.idxmin(axis=1)

chosen = choose(grp)                      # uses all development origins: this is what the final forecast uses
tab = grp.groupby("group").agg(SKUs_per_origin=("n", "mean"), picks_per_origin=("actual", "mean"))
for meth in ["Last 28 days", "LightGBM ratio"] + MENU:
    tab[meth] = grp.groupby("group")[meth].sum() / grp.groupby("group")["actual"].sum()
tab = tab.reindex(GROUPS)
tab["chosen"] = chosen.reindex(GROUPS)
print("WAPE by group (summed over the development origins). Chosen = lowest error in that row:")
display(tab[["SKUs_per_origin", "picks_per_origin", "Last 28 days", "LightGBM ratio", "chosen"] + MENU].round(3))

# leave-one-origin-out score of the per-group choice
loo_err, loo_act, base_err = 0.0, 0.0, 0.0
loo_rows = []
for e in STORE:
    ch = choose(grp[grp.origin != e]); s = STORE[e]; p = np.zeros(len(s["a"]))
    for g_ in GROUPS:
        m = s["group"] == g_; p[m] = s["preds"][ch[g_]][m]
    r = score_one(p, s["a"], s["base"]); r.update(origin=dates[e], method="Per-group choice (leave-one-out)"); loo_rows.append(r)
loo = pd.DataFrame(loo_rows)
cmp_ = pd.concat([summary[cols_show].loc[["Last 28 days", "LightGBM ratio", "Blend 0.50"]], loo.groupby("method")[cols_show].mean()])
cmp_["beats Last 28 (share of origins)"] = [
    (dev[dev.method == m].set_index("origin").WAPE < l28).mean() if m in set(dev.method) else (loo.set_index("origin").WAPE < l28).mean() for m in cmp_.index]
print("\nHonest comparison (the last row never saw the origin it is scored on):"); display(cmp_)

fig, ax = plt.subplots(figsize=(9, 4))
for m, st in (("Last 28 days", "-"), ("LightGBM ratio", "--"), ("Blend 0.50", ":")):
    d = dev[dev.method == m].sort_values("origin"); ax.plot(d.origin, d.WAPE, st, marker="o", label=m)
d = loo.sort_values("origin"); ax.plot(d.origin, d.WAPE, "-", marker="s", lw=2, label="Per-group choice (leave-one-out)")
ax.set_ylabel("SKU-level WAPE, next 28 days"); ax.set_xlabel("forecast origin"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Development origins: error by origin (lower is better)"); plt.tight_layout(); plt.show()

## 7. Locked test (optional, run once)

Leave `RUN_LOCKED_TEST = False` while you are still changing things. When settled, set it to `True` and run this cell **once**. It trains on everything known before the test window and scores the chosen per-group methods and the baselines on the final 28 days.

In [ ]:
def apply_choice(preds, group, ch):
    p = np.zeros(len(group))
    for g_ in GROUPS:
        m = group == g_; p[m] = preds[ch[g_]][m]
    return p

locked_table = None
if RUN_LOCKED_TEST:
    m_lock = fit_models(test_origin); fe = FE[test_origin]; a = outcome(test_origin)[fe["el"]]
    pr_ = all_predictions(test_origin, m_lock)
    pr_["Per-group choice"] = apply_choice(pr_, fe["group"], chosen)
    keep = ["Last 28 days", "LightGBM ratio", "Blend 0.50", "Per-group choice"]
    locked_table = pd.DataFrame({m: score_one(pr_[m], a, fe["base"]) for m in keep}).T
    print("LOCKED TEST, origin", dates[test_origin].date()); display(locked_table)
else:
    print("Locked test not run (RUN_LOCKED_TEST = False).")

## 8. The real forecast: the next month

The model is now trained on **every** origin whose outcome is fully known (including the former locked window) and forecasts from the last day in the data, using each group's chosen method. Per SKU:

* `expected_picks_28d`: the forecast pick count; `last28_picks`: the plain baseline; `method_used`: which method its group uses.
* `picks_low_80` / `picks_high_80`: an 80% range from how wrong the same method was for SKUs of that group and size in the development origins (the ranges are in-sample, so treat them as approximate).
* `rank` and `tier` (A = SKUs making up the first 80% of expected picks, B = next 15%, C = rest).
* `change_flag`: `up` / `down` when the forecast differs from last 28 days by at least `MOVER_THRESHOLD` and the SKU had at least `MOVER_MIN_PICKS` picks. These are the SKUs where the forecast says something last month does not.
* `case_share_28d` and `expected_case_picks_28d`: the share of the SKU's recent picks that were pick-face case picks, and the expected count of those. This is the part the slotting algorithm counts as visits.

In [ ]:
# ---- error-ratio quantiles for the ranges: by group x size of the forecast
def bucket(f): return np.where(f >= 500, "500+", np.where(f >= 100, "100-500", np.where(f >= 20, "20-100", "<20")))
rat_rows = []
for e, s in STORE.items():
    p = apply_choice(s["preds"], s["group"], chosen); m = p >= 1
    rat_rows.append(pd.DataFrame({"group": s["group"][m], "bucket": bucket(p[m]), "ratio": s["a"][m] / p[m]}))
rr = pd.concat(rat_rows)
qs = rr.groupby(["group", "bucket"])["ratio"].quantile(list(INTERVAL)).unstack().rename(columns={INTERVAL[0]: "lo", INTERVAL[1]: "hi"})
qs["n"] = rr.groupby(["group", "bucket"]).size()
qs.loc[qs.n < 100, ["lo", "hi"]] = np.nan                                  # too few observations: fall back below
# in-sample coverage check: how often did the real count land inside the range?
chk = rr.merge(qs[["lo", "hi"]].reset_index(), on=["group", "bucket"], how="left")
chk = chk.dropna(subset=["lo"]); chk["inside"] = (chk.ratio >= chk.lo) & (chk.ratio <= chk.hi)
print("Share of development SKU-forecasts that landed inside their 80% range (target 0.80):")
display(chk.groupby("group")["inside"].mean().reindex(GROUPS).round(3).to_frame("inside range"))
fallback = rr.groupby("group")["ratio"].quantile(list(INTERVAL)).unstack().rename(columns={INTERVAL[0]: "lo", INTERVAL[1]: "hi"})

models_final = fit_models(final_origin)
fe = FE[final_origin]; el = fe["el"]
lvl, rat = predict_models(models_final, fe)
pr_final = all_predictions(final_origin, models_final)
final = apply_choice(pr_final, fe["group"], chosen)
method_used = np.array([chosen[g_] for g_ in fe["group"]])

fc = pd.DataFrame({"item_id": skus.to_numpy()[el], "origin_date": dates[final_origin], "horizon_days": H,
                   "expected_picks_28d": final, "last28_picks": fe["base"], "group": fe["group"], "method_used": method_used,
                   "lightgbm_ratio_picks": rat, "active_rate": fe["rate"], "weekly_cv": fe["X"]["WEEKLY_CV"].to_numpy(), "source": SOURCE_TAG})
b = bucket(fc.expected_picks_28d.to_numpy())
lo = np.array([qs["lo"].get((g_, bk), np.nan) for g_, bk in zip(fc.group, b)]); hi = np.array([qs["hi"].get((g_, bk), np.nan) for g_, bk in zip(fc.group, b)])
lo = np.where(np.isnan(lo), fc.group.map(fallback["lo"]).to_numpy(), lo); hi = np.where(np.isnan(hi), fc.group.map(fallback["hi"]).to_numpy(), hi)
fc["picks_low_80"] = np.round(fc.expected_picks_28d * lo, 1); fc["picks_high_80"] = np.round(fc.expected_picks_28d * hi, 1)
fc = fc.sort_values("expected_picks_28d", ascending=False).reset_index(drop=True)
fc["rank"] = np.arange(1, len(fc) + 1)
share = fc.expected_picks_28d.cumsum() / max(fc.expected_picks_28d.sum(), 1e-9)
fc["tier"] = np.where(share <= 0.80, "A", np.where(share <= 0.95, "B", "C")); fc.loc[0, "tier"] = "A"
chg = fc.expected_picks_28d / fc.last28_picks.clip(lower=1e-9) - 1
fc["change_vs_last28"] = chg.where(fc.last28_picks > 0)
fc["change_flag"] = np.where((fc.last28_picks >= MOVER_MIN_PICKS) & (chg >= MOVER_THRESHOLD), "up",
                      np.where((fc.last28_picks >= MOVER_MIN_PICKS) & (chg <= -MOVER_THRESHOLD), "down", ""))
cs = (fe["X"]["CASE28"] / fe["X"]["T28"].replace(0, np.nan)).fillna(0).to_numpy()
fc["case_share_28d"] = pd.Series(cs, index=skus.to_numpy()[el]).reindex(fc.item_id).to_numpy().round(3)
fc["expected_case_picks_28d"] = (fc.expected_picks_28d * fc.case_share_28d).round(1)

print(f"Forecast from {dates[final_origin].date()}: {len(fc):,} SKUs | total expected picks {fc.expected_picks_28d.sum():,.0f} "
      f"(last 28 days: {fc.last28_picks.sum():,.0f}) | expected case picks {fc.expected_case_picks_28d.sum():,.0f}")
print("Tier A SKUs:", int((fc.tier == 'A').sum()), "| flagged up:", int((fc.change_flag == 'up').sum()), "| flagged down:", int((fc.change_flag == 'down').sum()))
print("\nMethod by group:", dict(chosen.reindex(GROUPS)))
display(fc.groupby("group").agg(SKUs=("item_id", "count"), expected_picks=("expected_picks_28d", "sum"), last28=("last28_picks", "sum")).reindex(GROUPS).round(0))
display(fc.head(8))

## 9. Save

In [ ]:
dev.to_csv(OUTPUT / "dev_scores_by_origin.csv", index=False)
summary.to_csv(OUTPUT / "dev_summary_by_method.csv")
tab.to_csv(OUTPUT / "wape_by_group_and_method.csv")
cmp_.to_csv(OUTPUT / "honest_comparison_leave_one_out.csv")
qs.reset_index().to_csv(OUTPUT / "interval_ratio_quantiles.csv", index=False)
fc.to_csv(OUTPUT / "forecast_sku_next_28d_picks.csv", index=False)
if locked_table is not None: locked_table.to_csv(OUTPUT / "locked_test_scores.csv")
joblib.dump(models_final["ratio"], OUTPUT / "lightgbm_ratio_final.joblib")
config = {"TARGET_COL": TARGET_COL, "HORIZON_DAYS": H, "MIN_HISTORY": MIN_HISTORY, "N_EVAL_ORIGINS": N_EVAL_ORIGINS, "EVAL_STEP": EVAL_STEP,
          "groups": {"RARE_MAX_ACTIVE_RATE": RARE_MAX_ACTIVE_RATE, "CV_STEADY": CV_STEADY, "CV_VARIABLE": CV_VARIABLE, "CV_WEEKS": CV_WEEKS},
          "W_GRID": list(W_GRID), "method_by_group": {k: str(v) for k, v in chosen.items()}, "LGB_PARAMS": LGB_PARAMS,
          "DROP_FEATURES": DROP_FEATURES, "seasonal_lag_days": SEASONAL_LAG_DAYS, "calendar_features": CALENDAR_FEATURES,
          "features_off_for_lack_of_data": dead, "forecast_origin": str(dates[final_origin].date()),
          "locked_test_origin": str(dates[test_origin].date()), "locked_test_run": bool(RUN_LOCKED_TEST), "source": SOURCE_TAG}
(OUTPUT / "run_config.json").write_text(json.dumps(config, indent=2, default=str))
print("Saved to:", OUTPUT)

## 10. How to read this, and what to do next

1. **Does anything beat "Last 28 days"?** Check WAPE *and* the share of origins where it wins. If the per-group choice (leave-one-out) beats it on most origins, the model is adding something; if not, last month is the honest answer.
2. **Which groups gain?** Expect Steady SKUs to be near-identical to last month (little to learn) and the gains, if any, to come from Variable and Erratic. A Rare SKU is mostly luck at a monthly horizon; the table shows which simple method does least badly there.
3. **Use the range, not just the number.** For Erratic and Rare SKUs the 80% range is wide, and that is the real message: their pick count next month cannot be called accurately, and a slotting decision should not lean on it.
4. **Two pick counts.** `expected_picks_28d` counts every pick. The slotting algorithm only counts pick-face case picks at mapped locations, so for that use `expected_case_picks_28d` (or tell me and I will forecast the case-pick count directly with the same method).
5. **When the second year arrives:** re-run the preprocessing and this notebook unchanged. The last-year features switch on, and you can try `CALENDAR_FEATURES = True`. Seasonality is the main thing one year of history cannot teach the model.
6. **Next ideas, in order of expected payoff:** order-level data (open orders and order-received dates) as leading features; the SKUs picked together on the same order (what the slotting detour actually depends on); more development origins once you have more history.

**Caveats:** these are pick counts, not customer demand; stockouts can depress historical picks; zero-fill after a SKU's last pick is an assumption; the LightGBM settings are reasonable defaults and are not tuned; the group thresholds (0.4 / 0.8 / 5%) are design choices you can change in the settings cell; 12 origins can show a difference of a point or two, not a few tenths.